# CodeSentinel full CodeBERT training
Full training requires dataset/network access and preferably a Colab GPU. SMOKE is separate pipeline verification. Save to Drive throughout; do not report earlier run metrics as reproduced. Review dataset licenses before downloading.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import subprocess
REPO_URL = "https://github.com/rudiously/CodeSentinel.git"
REPO_REF = "main"  # Prefer an immutable reviewed commit SHA.
assert REPO_URL.startswith("https://github.com/")
subprocess.run(["git", "clone", REPO_URL, "/content/CodeSentinel"], check=True)
subprocess.run(["git", "-C", "/content/CodeSentinel", "checkout", REPO_REF], check=True)
%cd /content/CodeSentinel
%pip install -r ml/requirements.txt -r backend/requirements.txt


In [ ]:
!python -m ml.data.download
!python -m ml.data.split


In [ ]:
from pathlib import Path
import yaml
config = yaml.safe_load(Path("ml/configs/train_codebert.yaml").read_text())
config["output"] = "/content/drive/MyDrive/CodeSentinel/models/codesentinel-codebert/v1"
config["mlflow_uri"] = "sqlite:////content/drive/MyDrive/CodeSentinel/mlflow.db"
Path(config["output"]).parent.mkdir(parents=True, exist_ok=True)
Path("/content/full-training.yaml").write_text(yaml.safe_dump(config))


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
!python -m ml.train --config /content/full-training.yaml


Training selects early stopping and calibration on validation only, then evaluates test exactly with frozen model/calibration. Best checkpoints are saved under Drive models/codesentinel-codebert/checkpoints/. Final exported artifact contains model.pt, tokenizer/, evaluation.json, metadata.json. TorchScript/PyTorch parity must pass. No ONNX claim.


In [ ]:
import json
artifact = Path(config["output"])
metadata = json.loads((artifact / "metadata.json").read_text())
assert metadata["artifact_kind"] == "TRAINED"
print("Durable artifact:", artifact)
print("Copy the entire v1 directory to local models/codesentinel-codebert/v1/")
print("Then set MODEL_ARTIFACT_DIR=models/codesentinel-codebert/v1 and restart model_service.")
print("Run: python -m ml.evaluation_run --artifact models/codesentinel-codebert/v1 --test ml/data/processed/test.jsonl --validation ml/data/processed/validation.jsonl --output docs/evaluation/trained-results.json")
